In [1]:
from fonctions.runes import Rune
from fonctions.gestion_bdd import lire_bdd, lire_bdd_perso
import json
import pandas as pd


2024-04-14 02:20:28.504 
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            
2024-04-14 02:20:28.506 
  command:

    streamlit run e:\PycharmProjects\SW\api\.venv\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2024-04-14 02:20:28.923 No runtime found, using MemoryCacheStorageManager


In [103]:
df = pd.read_excel('test.xlsx')

In [104]:
df_vio_will = df[df['rune_set'].isin(['Violent', 'Will'])]
df_vio_will

,Unnamed: 0,rune_set,rune_slot,rune_equiped,stars,qualité,qualité_original,level,efficiency,max_efficiency,...,fourth_sub_grinded_value,first_sub_value_max,second_sub_value_max,third_sub_value_max,fourth_sub_value_max,innate_value_max,first_sub_value_total,second_sub_value_total,third_sub_value_total,fourth_sub_value_total
532,14928895013,Violent,1,Inventaire,6,LGD,LGD,15,98.55,0,...,6,30,35,30,40,40,14,12,6,23
533,16197507534,Violent,1,Inventaire,6,LGD,HEROIQUE,12,76.99,0,...,522,30,30,40,3750,0,16,4,11,803
534,17490291881,Violent,1,Inventaire,6,LGD,RARE,15,88.86,0,...,6,35,30,30,40,40,6,20,6,14
535,20519695605,Violent,1,Inventaire,6,LGD,HEROIQUE,15,83.29,0,...,0,35,40,30,30,0,16,11,12,6
536,22106425139,Violent,1,Inventaire,6,LGD,HEROIQUE,15,89.58,0,...,0,40,30,30,40,0,15,19,9,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,22149241043,Violent,4,Mellia,6,LGD,LGD,15,94.81,0,...,0,35,30,40,30,0,13,18,14,10
1919,24992956338,Violent,5,Mellia,6,LGD,LGD,12,84.94,0,...,0,30,40,30,40,200,9,12,10,14
1920,21929836914,Violent,6,Mellia,6,LGD,HEROIQUE,15,77.86,0,...,0,30,200,40,40,0,6,31,26,7
1922,35918004252,Will,2,Altaïr,6,LGD,HEROIQUE,15,72.96,0,...,0,35,40,40,40,0,12,10,11,7


In [105]:
def detect_speed(df):
    for sub in ['first_sub', 'second_sub', 'third_sub', 'fourth_sub']:
        if df[sub] == 'SPD':  # stat speed = 8
            df['spd'] = df[f'{sub}_value_total']

    return df

In [106]:
data_spd = df_vio_will.apply(detect_speed, axis=1)

In [111]:
data_spd.loc[data_spd['rune_slot'] == 2] = 42

In [98]:
result = pd.DataFrame(data_spd.groupby(['rune_set', 'rune_slot'])['spd'].nlargest(1).reset_index()).drop(columns='level_2')

In [99]:
result

,rune_set,rune_slot,spd
0,Violent,1,26.0
1,Violent,2,25.0
2,Violent,3,28.0
3,Violent,4,31.0
4,Violent,5,29.0
5,Violent,6,28.0
6,Will,1,27.0
7,Will,2,32.0
8,Will,3,27.0
9,Will,4,26.0


In [100]:
import itertools

# Séparer les données par type
df_A = result[result['rune_set'] == 'Violent']
df_B = result[result['rune_set'] == 'Will']

# Créer toutes les combinaisons possibles de 4 lignes de type 'A' et 2 lignes de type 'B'
combinations_A = list(itertools.combinations(df_A.iterrows(), 4))
combinations_B = list(itertools.combinations(df_B.iterrows(), 2))

# Initialisation de la valeur maximale
max_value = 0
best_combination = None

# Parcourir chaque combinaison
for comb_A in combinations_A:
    for comb_B in combinations_B:
        # Extraire les lignes de chaque combinaison
        selected_rows = [row[1] for row in comb_A] + [row[1] for row in comb_B]
        
        # Vérifier si chaque slot est unique
        slots = set()
        unique_slots = True
        for row in selected_rows:
            if row['rune_slot'] in slots:
                unique_slots = False
                break
            slots.add(row['rune_slot'])
        
        # Si tous les slots sont uniques, calculer la somme des valeurs pour cette combinaison
        if unique_slots:
            total_value = sum(row['spd'] for row in selected_rows)
            
            # Mettre à jour la valeur maximale et la meilleure combinaison
            if total_value > max_value:
                max_value = total_value
                best_combination = selected_rows

df_max_spd = pd.DataFrame(best_combination)
value_max = df_max_spd['spd'].sum()

In [101]:
df_max_spd

,rune_set,rune_slot,spd
2,Violent,3,28.0
3,Violent,4,31.0
4,Violent,5,29.0
5,Violent,6,28.0
6,Will,1,27.0
7,Will,2,32.0
